# Black-Litterman Portfolio with LSTM + SVR Investor's Views

**Architecture B** — one multi-output LSTM per stock (predicts all 8 indicators at once),
then an SVR decodes the predicted indicators into a price forecast. The forecast becomes
the Black-Litterman view (Q), and the model's error sets the view's uncertainty (Ω).

```
yfinance OHLCV → 8 indicators → 30-day windows
   → LSTM (1 per stock, 8 outputs) → SVR (indicators → price)
   → views Q + Ω → Black-Litterman w*  → backtest vs SPY
```

This notebook **reuses the verified `bl_lstm_views/src` modules** (so the heavy logic is
tested), and keeps the project's two refinements over the raw design doc:

- **Stationary price target** — the SVR predicts `close / SMA₃₀(close) − 1`, not the raw
  price level, so an RBF model can't be wrecked by the 2010→2025 price trend.
- **Long-only + capped, renormalised weights** and the **no-views ⇒ w\*=w_mkt** identity.

> Research scaffold, **not** a profitable strategy — backtest numbers are illustrative.
> See `README.md` for the full caveats.

In [ ]:
# Fresh setup:  pip install -r requirements.txt   (numpy pandas scikit-learn yfinance torch)
import os, sys, warnings, json
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import torch

sys.path.insert(0, os.getcwd())          # run this notebook from the bl_lstm_views/ folder
from src import constants as C
from src import fetch_data, black_litterman as bl
from src.indicators import add_indicators
from src.preprocessing import build_stock_dataset
from src.lstm_model import LSTMRegressor, get_device, train_stock_model, predict_indicators
from src.svr_model import train_close_svr, predict_price, train_fit_mse
from src.views import compose_view, assemble_package, write_views_json
from src.backtest import run_backtest

TICKERS    = ["RKLB","AMAT", "CRWD", "RBRK", "IONQ", "LLY", "EOSE", "ASML", "ELV", "PLTR", "OKLO", "NVDA", "MSFT"]
EPOCHS     = 400
MIN_WEIGHT = 0.01      # ← ปรับตรงนี้ (ขั้นต่ำที่ถือแต่ละตัว, 0.00 = ไม่มีขั้นต่ำ)
MAX_WEIGHT = 0.40

# ── LSTM hyperparams — auto-loaded from Optuna if available ──────────────────
_best_params_path = os.path.join("output", "best_lstm_params.json")
if os.path.exists(_best_params_path):
    _bp = json.load(open(_best_params_path))
    LSTM_LR    = _bp["learning_rate"]
    LSTM_BATCH = _bp["batch_size"]
    print(f"✅ Optuna best params loaded: LR={LSTM_LR}, BATCH={LSTM_BATCH} "
          f"(mean_val_MSE={_bp.get('val_mse_mean', float('nan')):.4f})")
else:
    LSTM_LR    = C.LSTM_LR
    LSTM_BATCH = C.LSTM_BATCH
    print(f"ℹ️  Optuna params not found — using defaults: LR={LSTM_LR}, BATCH={LSTM_BATCH}")

device = get_device(os.environ.get("BL_DEVICE", "auto"))
pct = lambda x: np.round(np.asarray(x, float) * 100, 2)
print("torch", torch.__version__, "| device:", device.type)
print("tickers:", TICKERS, "| epochs:", EPOCHS, "| min_weight:", MIN_WEIGHT)

## Step 1 — Fetch Data
Daily OHLCV + market caps from yfinance (cached to `output/cache/`).

In [2]:
data   = fetch_data.fetch(TICKERS)
prices = data["prices"]
caps   = data["market_caps"]
bench  = fetch_data.download_prices(C.BENCHMARK, C.DATA_START, C.DATA_END)
bench_close = bench["close"] if bench is not None else None

# ── summary table ────────────────────────────────────────────────────────
df_summary = pd.DataFrame({t: {"Days": len(df), "Market Caps": round(caps.get(t, 0) / 1e9, 2)}
                           for t, df in prices.items()}).T

total_caps = df_summary["Market Caps"].sum()
df_summary["W_mkt"] = round(df_summary["Market Caps"] / total_caps, 4)
df_summary.loc["Total"] = ["", total_caps, df_summary["W_mkt"].sum()]
display(df_summary)

# ── skipped at fetch stage (insufficient history or download failure) ────
skipped = dict(data["errors"])  # carried through to final summary cell
if skipped:
    print(f"\n⚠️  {len(skipped)} ticker(s) dropped at fetch stage (before any modelling):")
    for t, reason in sorted(skipped.items()):
        print(f"   ✂️  {t}: {reason}")


  ↗  Fetching ['RKLB', 'AMAT', 'CRWD', 'RBRK', 'IONQ', 'LLY', 'EOSE', 'ASML', 'ELV', 'PLTR', 'OKLO', 'NVDA', 'MSFT']  (2010-01-01 → 2026-06-24)
      (minimum 878 days required: 500 train + 126 val + 252 test)
  ✅  RKLB: 1399 daily bars
  ✅  AMAT: 4142 daily bars
  ✅  CRWD: 1767 daily bars
  ✅  RBRK: 541 daily bars
  ✂️   RBRK: dropped — insufficient history: 541 days < 878 required (500 train + 126 val + 252 test)
  ✅  IONQ: 1373 daily bars
  ✅  LLY: 4142 daily bars
  ✅  EOSE: 1415 daily bars
  ✅  ASML: 4142 daily bars
  ✅  ELV: 4142 daily bars
  ✅  PLTR: 1438 daily bars
  ✅  OKLO: 1245 daily bars
  ✅  NVDA: 4142 daily bars
  ✅  MSFT: 4142 daily bars
  ✅  SPY: 4142 daily bars


,Days,Market Caps,W_mkt
RKLB,1399.0,53.37,0.0052
AMAT,4142.0,467.62,0.0454
CRWD,1767.0,171.33,0.0167
IONQ,1373.0,20.01,0.0019
LLY,4142.0,996.31,0.0968
EOSE,1415.0,2.06,0.0002
ASML,4142.0,679.40,0.0660
ELV,4142.0,83.39,0.0081
PLTR,1438.0,272.09,0.0264
OKLO,1245.0,9.41,0.0009



⚠️  1 ticker(s) dropped at fetch stage (before any modelling):
   ✂️  RBRK: insufficient history: 541 days < 878 required (500 train + 126 val + 252 test)


## Step 2 — Compute Indicators (8)
MACD · RSI · squeeze_on · BB_width · KC_width · sq_momentum · OBV · Hilbert — all pure pandas/numpy (`src/indicators.py`).

In [3]:
sample = add_indicators(prices[TICKERS[0]])[C.FEATURES].dropna()
print(f"{TICKERS[0]}: {sample.shape[0]} rows x {sample.shape[1]} features")
sample.tail()

RKLB: 1369 rows x 8 features


Price,macd,rsi,squeeze_on,bb_width,kc_width,sq_momentum,obv,hilbert
Date,,,,,,,,
2026-06-16,-0.426203,45.335988,0,59.340184,33.456836,-18.053436,-0.039315,30.0
2026-06-17,-0.774367,47.326375,0,60.547442,32.356183,-17.109809,-0.021842,30.0
2026-06-18,-1.097353,46.919962,0,60.692143,31.728929,-15.552496,-0.058909,30.0
2026-06-22,-1.892314,43.170365,0,62.967816,31.821412,-16.412938,-0.071629,30.0
2026-06-23,-2.906004,40.572879,0,64.384935,30.746516,-17.015826,-0.079770,30.0


## Step 3 — Preprocess
Train-only MinMax scaling, 30-day windows, 5-day-ahead targets (`src/preprocessing.py`).  
**Hybrid split** (fixed from the end of each stock's history):
- **Test**: last 252 days (≈ 1 year) — never seen during training
- **Val**: prev 126 days (≈ 6 months) — used only for early stopping / LR schedule
- **Train**: everything before that

Stocks with < 878 total days (500 train + 126 val + 252 test) are rejected at fetch and excluded here.


In [4]:
stock_data = {}
for t in TICKERS:
    if t not in prices:
        continue   # already in skipped from fetch stage
    try:
        stock_data[t] = build_stock_dataset(t, prices[t])
    except Exception as e:
        print(f"⚠️  {t}: skipped at preprocessing ({e})")
        skipped[t] = str(e)

sd0 = stock_data[list(stock_data.keys())[0]]; w = sd0.windows
print("X:", w["X"].shape, " Y:", w["Y"].shape,
      " (window=", C.TIME_WINDOW, ", features=", C.LSTM_OUTPUTS, ", horizon=", C.HORIZON, ")")
print("windows  train/val/test:", int(w["train"].sum()), int(w["val"].sum()), int(w["test"].sum()))


X: (1335, 30, 8)  Y: (1335, 8)  (window= 30 , features= 8 , horizon= 5 )
windows  train/val/test: 962 126 247


## Step 4 — Train LSTM (Architecture B)
One 4-layer LSTM per stock, `Dense(60 → 8)` multi-output head. Adam(5e-3), dropout 0.30, val-driven early stopping + LR plateau.

In [5]:
demo = LSTMRegressor()
print(demo)
print(f"\n{sum(p.numel() for p in demo.parameters()):,} params per stock"
      f"  ->  {len(stock_data)} models total (Architecture B)")

LSTMRegressor(
  (lstm): LSTM(8, 60, num_layers=4, batch_first=True, dropout=0.3)
  (head): Linear(in_features=60, out_features=8, bias=True)
)

105,128 params per stock  ->  12 models total (Architecture B)


In [ ]:
lstm_results = {}
for t, sd in stock_data.items():
    print(f"\nTraining {t} ...")
    lstm_results[t] = train_stock_model(sd, epochs=EPOCHS, lr=LSTM_LR, batch=LSTM_BATCH, device=device)
print("\nLSTM training done.")

records = {}
for t, r in lstm_results.items():
    val = r["val_mse"]
    test = float(np.nanmean(r["mse_test"]))
    records[t] = {
        "val_MSE": round(val, 4),
        "test_MSE_mean": round(test, 4),
        "change x": round((test / val), 2)
    }
pd.DataFrame(records).T

## Step 5 — Train SVR
SVR(RBF, C=5, ε=0.01) maps scaled indicators → `close/SMA₃₀ − 1` (the stationary target). At inference it decodes the LSTM's forecast into a price.

In [7]:
svr_models = {t: train_close_svr(sd) for t, sd in stock_data.items()}

# ── SVR diagnostics: train vs test ──────────────────────────────────────
rows = {}
for t, sd in stock_data.items():
    svr = svr_models[t]
    # training fit
    fit_mse = train_fit_mse(svr, sd)
    # test performance with ACTUAL indicators (upper-bound; LSTM adds noise in practice)
    X_te = sd.feats_scaled[sd.test_mask]
    y_te = sd.close_rel[sd.test_mask]
    pred_te = svr.predict(X_te)
    test_mse = float(((pred_te - y_te) ** 2).mean())
    naive_mse = float((y_te ** 2).mean())  # predict-zero baseline
    r2 = 1 - test_mse / naive_mse if naive_mse > 0 else float("nan")
    dir_acc = float((np.sign(pred_te) == np.sign(y_te)).mean())  # direction accuracy
    rows[t] = {
        "train_MSE":   round(fit_mse,  6),
        "test_MSE":    round(test_mse, 6),
        "test_RMSE_%": round(test_mse**0.5 * 100, 2),
        "test_R2":     round(r2, 3),
        "dir_acc_%":   round(dir_acc * 100, 1),
        "n_test_days": int(sd.test_mask.sum()),
    }

df_svr = pd.DataFrame(rows).T
display(df_svr)
print("\nNote: test metrics use actual indicators (not LSTM predictions)."
      " Real pipeline performance is lower due to LSTM forecast noise.")


,train_MSE,test_MSE,test_RMSE_%,test_R2,dir_acc_%,n_test_days
RKLB,0.000615,0.046801,21.63,-0.019,64.3,252.0
AMAT,0.000338,0.011727,10.83,-0.008,75.0,252.0
CRWD,0.000927,0.009998,10.00,0.134,89.7,252.0
IONQ,0.001094,0.048685,22.06,-0.347,84.5,252.0
LLY,0.000117,0.001074,3.28,0.791,89.3,252.0
EOSE,0.002890,0.020943,14.47,0.633,91.7,252.0
ASML,0.000275,0.028398,16.85,-2.580,65.5,252.0
ELV,0.000163,0.000792,2.81,0.885,94.8,252.0
PLTR,0.001705,0.048478,22.02,-6.224,60.3,252.0
OKLO,0.000205,0.051709,22.74,-0.340,55.6,252.0



Note: test metrics use actual indicators (not LSTM predictions). Real pipeline performance is lower due to LSTM forecast noise.


## Step 6 — Black-Litterman
Compose each stock's view (`compose_view`), build Π, μ_BL and the optimal weights w\* (`src/black_litterman.py`), and export `views.json`.

In [ ]:
per_stock = {t: compose_view(stock_data[t], lstm_results[t], svr_models[t]) for t in stock_data}
views     = assemble_package(per_stock)
tickers   = views["tickers"]

panel = pd.DataFrame({t: pd.Series(stock_data[t].close_raw,
                      index=pd.DatetimeIndex(stock_data[t].dates).normalize())
                      for t in tickers}).dropna()
sigma = bl.covariance(panel.pct_change().dropna().to_numpy())
w_mkt = bl.market_weights(np.array([caps.get(t, 0) for t in tickers]))
res   = bl.solve(sigma, w_mkt, P=views["P"], Q=views["Q"], omega=views["omega"],
                 max_weight=MAX_WEIGHT, min_weight=MIN_WEIGHT)

write_views_json(views)
print("views.json ->", os.path.relpath(C.VIEWS_JSON))

pd.DataFrame({
    "view_q_5d_%": pct([per_stock[t]["q_horizon"] for t in tickers]),
    "Pi_ann_%":    pct(res["pi"]),
    "muBL_ann_%":  pct(res["mu_bl"]),
    "w_mkt_%":     pct(w_mkt),
    "w_star_%":    pct(res["w_star"]),
    "dW_%":        pct(res["w_star"] - w_mkt),
}, index=tickers)

## Step 7 — Evaluate
Rolling 5-day-rebalance backtest vs SPY (annual return, Sharpe, max drawdown), plus the weight tilt the views produced.

In [ ]:
bt = run_backtest(stock_data, views, w_mkt, bench_close,
                  max_weight=MAX_WEIGHT, min_weight=MIN_WEIGHT)
s, b = bt["strategy"], bt["benchmark"]
print(f"backtest grid {bt['grid_start']} -> {bt['grid_end']}  ({bt['n_rebalances']} rebalances)")

metrics = pd.DataFrame(
    {"BL+model": s, "SPY_buy_hold": b}).T[["ann_return", "ann_vol", "sharpe", "max_drawdown"]]
display(metrics.round(4))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
x, bw = np.arange(len(tickers)), 0.38
ax[0].bar(x - bw/2, pct(w_mkt), bw, label="w_mkt")
ax[0].bar(x + bw/2, pct(res["w_star"]), bw, label="w*")
ax[0].set_xticks(x); ax[0].set_xticklabels(tickers); ax[0].set_ylabel("%")
ax[0].set_title("Market vs BL optimal weights"); ax[0].legend()
ax[1].bar(["BL+model", "SPY"], [s["ann_return"]*100, b["ann_return"]*100], color=["#3b82f6", "#9ca3af"])
ax[1].set_ylabel("Annual return %"); ax[1].set_title("Backtest (test year)")
plt.tight_layout(); plt.show()

In [10]:
# ── Skipped tickers summary ─────────────────────────────────────────────
if skipped:
    print("=" * 60)
    print(f"  ⚠️  Skipped tickers ({len(skipped)} excluded from model):")
    print(f"  {'Ticker':<8}  Reason")
    print("  " + "-" * 58)
    for t, reason in sorted(skipped.items()):
        print(f"  {t:<8}  {reason}")
    print(f"\n  → {len(skipped)} skipped · {len(stock_data)} used in model.")
else:
    print(f"  → All {len(TICKERS)} tickers used in model (none skipped).")


  ⚠️  Skipped tickers (1 excluded from model):
  Ticker    Reason
  ----------------------------------------------------------
  RBRK      insufficient history: 541 days < 878 required (500 train + 126 val + 252 test)

  → 1 skipped · 12 used in model.
